# Study 943 — Reset Frequency — the teardown

The excess-of-cash race at 2x and 3x, the Newey-West difference *t*, paired block-bootstrap CIs on the Sharpe advantage, the efficiency-ratio decomposition (and the proof that it is arithmetic), the predictive version, the era cut, three sweeps — financing spread, cost, maintenance margin — the 2008 stress, and the live synthetic control.

Real numbers are frozen from `docs/results.md` (fingerprints `8f71ea1922ec` for the 2x sleeve, `4a95749c18bb` for the 3x). Construction: exposure `w` on the index funded at `^IRX + 50 bps`, `r_p = w·r − (w−1)·f − cost`, one execution lag on the reset, 2 bps one-way on |Δw|·NAV, 25% maintenance margin. Long-only, so there is no stock borrow; the only borrow is cash, and its spread is swept. A margin-called arm is credited the cash leg for the rest of the sample, so a call costs it the equity it destroyed and nothing more.

In [1]:
R = {'asof': '2026-06-30', 'spread_bps': 50, 'cost_bps': 2, 'maintenance': 25, 'x2_start': '2007-05-31', 'x2_end': '2026-06-30', 'x2_n': 4799, 'x2_fp': '8f71ea1922ec', 'x2_fund_sh': 0.504, 'x2_fund_cagr': 14.33, 'x2_fund_vol': 38.9, 'x2_fund_dd': -84.7, 'x2_fund_term': 12.8, 'x2_day_sh': 0.527, 'x2_day_cagr': 15.45, 'x2_day_vol': 39.6, 'x2_day_dd': -84.2, 'x2_day_term': 15.4, 'x2_mon_sh': 0.549, 'x2_mon_cagr': 17.13, 'x2_mon_vol': 42.8, 'x2_mon_dd': -82.5, 'x2_mon_term': 20.3, 'x2_spy_sh': 0.613, 'x2_spy_cagr': 10.7, 'x2_spy_vol': 19.8, 'x2_spy_dd': -55.2, 'x2_spy_term': 6.9, 'x2_diff_bps': 1.052, 'x2_t': 2.79, 'x2_adv': 0.022, 'x2_adv_ci_lo': -0.01, 'x2_adv_ci_hi': 0.062, 'x2_adv_frac_neg': 8.7, 'x2_mon_ci_lo': 0.168, 'x2_mon_ci_hi': 0.951, 'x2_day_ci_lo': 0.122, 'x2_day_ci_hi': 0.941, 'x2_vs_fund': 0.045, 'x2_t_vs_fund': 3.41, 'x2_fee_leg': 1.21, 'x2_t_fee': 3.82, 'x2_w_mean': 1.998, 'x2_w_min': 1.78, 'x2_w_max': 3.24, 'x2_months': 229, 'x2_slope': -1.25, 'x2_slope_t': -6.83, 'x2_chop': 0.381, 'x2_chop_t': 4.59, 'x2_trend': -0.164, 'x2_trend_t': -8.4, 'x2_pred_slope': 0.115, 'x2_pred_t': 0.72, 'x2_switch': 0.06, 'x2_switch_t': 2.49, 'x2_era_e_adv': 0.047, 'x2_era_e_t': 2.26, 'x2_era_l_adv': 0.004, 'x2_era_l_t': 1.56, 'x2_era_diff_bps': -0.836, 'x2_era_diff_t': -1.15, 'x3_era_diff_bps': -2.369, 'x3_era_diff_t': -0.3, 'x2_sp0_adv': 0.021, 'x2_sp200_adv': 0.025, 'x2_c0_adv': 0.021, 'x2_c10_adv': 0.029, 'x3_start': '2009-06-26', 'x3_end': '2026-06-30', 'x3_n': 4276, 'x3_fp': '4a95749c18bb', 'x3_fund_sh': 0.791, 'x3_fund_cagr': 32.95, 'x3_fund_vol': 51.3, 'x3_fund_dd': -76.8, 'x3_fund_term': 125.5, 'x3_day_sh': 0.808, 'x3_day_cagr': 34.17, 'x3_day_vol': 51.3, 'x3_day_dd': -76.2, 'x3_day_term': 146.6, 'x3_mon_sh': 0.239, 'x3_mon_cagr': 4.05, 'x3_mon_vol': 18.8, 'x3_mon_dd': -48.3, 'x3_mon_term': 2.0, 'x3_spy_sh': 0.911, 'x3_spy_cagr': 15.17, 'x3_spy_term': 11.0, 'x3_diff_bps': -14.678, 'x3_t': -3.62, 'x3_adv': -0.569, 'x3_adv_ci_lo': -1.085, 'x3_adv_ci_hi': 0.017, 'x3_fee_leg': 0.94, 'x3_t_fee': 3.71, 'x3_w_mean': 2.944, 'x3_w_mean_free': 2.983, 'x3_vol_free': 58.5, 'x3_dd_free': -82.2, 'x3_m30_adv': 0.124, 'x3_w_max_free': 8.42, 'x3_w_max_25': 3.84, 'x3_liq': '2011-08-08', 'x3_months': 204, 'x3_slope': -3.158, 'x3_slope_t': -7.22, 'x3_chop': 0.95, 'x3_chop_t': 3.91, 'x3_trend': -0.492, 'x3_trend_t': -8.29, 'x3_pred_slope': 0.659, 'x3_pred_t': 1.61, 'x3_switch': 0.041, 'x3_switch_t': 1.04, 'x3_era_e_adv': -0.581, 'x3_era_e_t': -2.8, 'x3_era_l_adv': -0.563, 'x3_era_l_t': -2.48, 'x3_m0_adv': 0.003, 'x3_m0_term': 227.3, 'x3_m15_adv': -0.285, 'x3_m15_liq': '2020-03-20', 'x3_m30_liq_daily': '2010-05-06', 'st_start': '2004-01-05', 'st_end': '2026-06-30', 'st_n': 5652, 'st_2x_mon': 37.56, 'st_2x_day': 27.91, 'st_3x_mon_free': 80.43, 'st_3x_mon_free_w': 12.8, 'st_3x_mon_25': 0.96, 'st_3x_liq': '2008-09-29', 'st_3x_day': 33.75, 'st_4x_mon_free': 0.0, 'st_4x_liq': '2008-10-24', 'st_4x_day': 17.37, 'syn_chop_gap': 0.357, 'syn_chop_t': 3.74, 'syn_chop_slope': -1.99, 'syn_trend_gap': -0.403, 'syn_trend_t': -2.47, 'syn_trend_slope': -2.65, 'syn_null_gap': 0.033, 'syn_null_t': 0.27, 'syn_null_slope': -2.29}

## 1. The 2x race, excess-of-cash

> 💡 **In plain words** — three ways to hold twice the S&P: buy the fund, do it yourself and rebalance nightly, or do it yourself and rebalance monthly.

In [2]:
for label, k in [('SSO (fund)', 'x2_fund'), ('daily-synth', 'x2_day'),
                 ('monthly-synth', 'x2_mon'), ('SPY 1x', 'x2_spy')]:
    print(f"{label:<14s} exSharpe {R[k+'_sh']:+.3f}  CAGR {R[k+'_cagr']:+6.2f}%  "
          f"vol {R[k+'_vol']:5.1f}%  maxDD {R[k+'_dd']:6.1f}%  terminal x{R[k+'_term']:.1f}")
print(f"\nmonthly - daily : {R['x2_diff_bps']:+.3f} bps/day  HAC t {R['x2_t']:+.2f}  "
      f"Sharpe advantage {R['x2_adv']:+.3f}")
print(f"paired block-bootstrap CI on the advantage: "
      f"[{R['x2_adv_ci_lo']:+.3f}, {R['x2_adv_ci_hi']:+.3f}]  "
      f"({R['x2_adv_frac_neg']:.1f}% of resamples negative)")
print(f"monthly Sharpe CI [{R['x2_mon_ci_lo']:+.3f}, {R['x2_mon_ci_hi']:+.3f}]  "
      f"daily Sharpe CI [{R['x2_day_ci_lo']:+.3f}, {R['x2_day_ci_hi']:+.3f}] -- superimposed")
print(f"exposure path (monthly arm): mean {R['x2_w_mean']:.3f}  min {R['x2_w_min']:.2f}  "
      f"max {R['x2_w_max']:.2f}")

SSO (fund)     exSharpe +0.504  CAGR +14.33%  vol  38.9%  maxDD  -84.7%  terminal x12.8
daily-synth    exSharpe +0.527  CAGR +15.45%  vol  39.6%  maxDD  -84.2%  terminal x15.4
monthly-synth  exSharpe +0.549  CAGR +17.13%  vol  42.8%  maxDD  -82.5%  terminal x20.3
SPY 1x         exSharpe +0.613  CAGR +10.70%  vol  19.8%  maxDD  -55.2%  terminal x6.9

monthly - daily : +1.052 bps/day  HAC t +2.79  Sharpe advantage +0.022
paired block-bootstrap CI on the advantage: [-0.010, +0.062]  (8.7% of resamples negative)
monthly Sharpe CI [+0.168, +0.951]  daily Sharpe CI [+0.122, +0.941] -- superimposed
exposure path (monthly arm): mean 1.998  min 1.78  max 3.24


The mean-return difference clears |*t*| = 2; the **Sharpe** difference does not. Mean exposure is 2.00 on the nose, so this is not a crude leverage mismatch at 2x — it is the path convexity, and it is worth two hundredths of a Sharpe.

## 2. Decomposing the gap versus the fund

> 💡 **In plain words** — beating SSO is two separate wins: rebalancing less often, and not paying SSO's fee. Only the first one is this study's subject.

In [3]:
print(f"monthly-synth - SSO        : {R['x2_vs_fund']:+.3f} Sharpe (HAC t {R['x2_t_vs_fund']:+.2f})")
print(f"  reset-frequency leg      : {R['x2_adv']:+.3f}")
print(f"  fee/tracking leg (daily-synth - SSO): {R['x2_fee_leg']:+.2f}%/yr "
      f"(HAC t {R['x2_t_fee']:+.2f})")
print(f"3x: fee/tracking leg (daily-synth - UPRO): {R['x3_fee_leg']:+.2f}%/yr "
      f"(HAC t {R['x3_t_fee']:+.2f})")

monthly-synth - SSO        : +0.045 Sharpe (HAC t +3.41)
  reset-frequency leg      : +0.022
  fee/tracking leg (daily-synth - SSO): +1.21%/yr (HAC t +3.82)
3x: fee/tracking leg (daily-synth - UPRO): +0.94%/yr (HAC t +3.71)


## 3. Trending versus choppy — the efficiency-ratio decomposition

Per month, regress the log gap (monthly − daily, pp) on ER = |Σ log r| / Σ|log r| on SPY. ER = 1 is a straight-line month; ER ≈ 0 is a round trip.

> 💡 **In plain words** — did the month go somewhere, or did it thrash about?

In [4]:
print(f"2x ({R['x2_months']} months): slope {R['x2_slope']:+.3f} pp per unit ER  "
      f"HAC t {R['x2_slope_t']:+.2f}")
print(f"   choppy tercile {R['x2_chop']:+.3f} pp (t {R['x2_chop_t']:+.2f})   "
      f"trending tercile {R['x2_trend']:+.3f} pp (t {R['x2_trend_t']:+.2f})")
print(f"3x ({R['x3_months']} months): slope {R['x3_slope']:+.3f} pp per unit ER  "
      f"HAC t {R['x3_slope_t']:+.2f}")
print(f"   choppy tercile {R['x3_chop']:+.3f} pp (t {R['x3_chop_t']:+.2f})   "
      f"trending tercile {R['x3_trend']:+.3f} pp (t {R['x3_trend_t']:+.2f})")
print()
print(f"LAGGED (tradable) version: 2x slope {R['x2_pred_slope']:+.3f} "
      f"(t {R['x2_pred_t']:+.2f}); 3x slope {R['x3_pred_slope']:+.3f} (t {R['x3_pred_t']:+.2f})")
print(f"naive switch rule (monthly after a choppy month, IN-SAMPLE median threshold): "
      f"2x {R['x2_switch']:+.3f} pp/mo (t {R['x2_switch_t']:+.2f}), "
      f"3x {R['x3_switch']:+.3f} pp/mo (t {R['x3_switch_t']:+.2f})")

2x (229 months): slope -1.250 pp per unit ER  HAC t -6.83
   choppy tercile +0.381 pp (t +4.59)   trending tercile -0.164 pp (t -8.40)
3x (204 months): slope -3.158 pp per unit ER  HAC t -7.22
   choppy tercile +0.950 pp (t +3.91)   trending tercile -0.492 pp (t -8.29)

LAGGED (tradable) version: 2x slope +0.115 (t +0.72); 3x slope +0.659 (t +1.61)
naive switch rule (monthly after a choppy month, IN-SAMPLE median threshold): 2x +0.060 pp/mo (t +2.49), 3x +0.041 pp/mo (t +1.04)


The contemporaneous slope is enormous and the sign is unambiguous — and **both are mechanical**. Section 7 measures the same slope of -2.29 on an *iid random walk*, where by construction there is nothing to find. A month that happened to be choppy always favours the monthly reset; that is the compounding algebra, not information. What is *not* mechanical is the unconditional mean gap, which is zero on the null and +1.05 bps/day on the real 2x tape — a leverage-lens restatement of the post-2000 negative daily autocorrelation of the S&P.

> 💡 **In plain words** — you cannot pick next month's reset frequency, because you cannot know in advance whether next month will trend.

## 4. The 3x sleeve and the maintenance-margin sweep

> 💡 **In plain words** — the answer at 3x depends entirely on how patient your lender is, so here is every level of patience.

In [5]:
for label, k in [('UPRO (fund)', 'x3_fund'), ('daily-synth', 'x3_day'), ('monthly-synth', 'x3_mon')]:
    print(f"{label:<14s} exSharpe {R[k+'_sh']:+.3f}  CAGR {R[k+'_cagr']:+6.2f}%  "
          f"terminal x{R[k+'_term']:.1f}")
print(f"monthly - daily: {R['x3_diff_bps']:+.2f} bps/day  HAC t {R['x3_t']:+.2f}  "
      f"Sharpe advantage {R['x3_adv']:+.3f}  CI [{R['x3_adv_ci_lo']:+.3f}, {R['x3_adv_ci_hi']:+.3f}]")
print()
print('maintenance-margin sweep (PROXY):')
print(f"   0%: advantage {R['x3_m0_adv']:+.3f}  peak exposure {R['x3_w_max_free']:.2f}x  "
      f"terminal x{R['x3_m0_term']:.1f}  never called")
print(f"  15%: advantage {R['x3_m15_adv']:+.3f}  called {R['x3_m15_liq']}")
print(f"  25%: advantage {R['x3_adv']:+.3f}  called {R['x3_liq']}  "
      f"terminal x{R['x3_mon_term']:.1f}")
print(f"  30%: even the DAILY-reset margin account is called ({R['x3_m30_liq_daily']}, "
      f"the flash crash)")

UPRO (fund)    exSharpe +0.791  CAGR +32.95%  terminal x125.5
daily-synth    exSharpe +0.808  CAGR +34.17%  terminal x146.6
monthly-synth  exSharpe +0.239  CAGR  +4.05%  terminal x2.0
monthly - daily: -14.68 bps/day  HAC t -3.62  Sharpe advantage -0.569  CI [-1.085, +0.017]

maintenance-margin sweep (PROXY):
   0%: advantage +0.003  peak exposure 8.42x  terminal x227.3  never called
  15%: advantage -0.285  called 2020-03-20
  25%: advantage -0.569  called 2011-08-08  terminal x2.0
  30%: even the DAILY-reset margin account is called (2010-05-06, the flash crash)


Read the 0% row as the fair-fight upper bound: with a lender who never calls, the 3x monthly reset's Sharpe advantage is **+0.003** — it converts a ×146.6 into a ×227.3 on mean exposure **2.98** — the daily arm's own 3.00 — by running 58.5% vol against 51.3% and a -82.2% drawdown against -76.2%. Same leverage, more risk, no Sharpe. The 30% row is the deeper point: at a broker requirement most retail accounts actually face **no** 3x margin account survives (the +0.124 there is two dead arms, not a win) — which is why these things are funds.

## 5. Era cut and the other two sweeps

> 💡 **In plain words** — does the answer depend on when you looked, on how expensive your borrowing is, or on your commissions? On this tape, none of the three: the halves look different but are not statistically distinguishable.

In [6]:
print(f"2x era cut: 2007-2016 {R['x2_era_e_adv']:+.3f} (t {R['x2_era_e_t']:+.2f})  |  "
      f"2017-2026 {R['x2_era_l_adv']:+.3f} (t {R['x2_era_l_t']:+.2f})")
print(f"  test of the DIFFERENCE (gap on an era dummy): "
      f"{R['x2_era_diff_bps']:+.3f} bps/day, HAC t {R['x2_era_diff_t']:+.2f} "
      f"-- NOT distinguishable, so no decay is claimed")
print(f"3x era cut: 2009-2016 {R['x3_era_e_adv']:+.3f} (t {R['x3_era_e_t']:+.2f})  |  "
      f"2017-2026 {R['x3_era_l_adv']:+.3f} (t {R['x3_era_l_t']:+.2f})   "
      f"difference {R['x3_era_diff_bps']:+.3f} bps/day "
      f"(t {R['x3_era_diff_t']:+.2f})")
print()
print(f"2x financing spread   0 bps: adv {R['x2_sp0_adv']:+.3f}   "
      f"200 bps: adv {R['x2_sp200_adv']:+.3f}")
print(f"2x trading cost       0 bps: adv {R['x2_c0_adv']:+.3f}   "
      f"10 bps one-way: adv {R['x2_c10_adv']:+.3f}")
print('(both arms hold nearly the same average exposure, so financing and cost\n'
      ' hit them almost identically -- neither sweep can rescue or kill the result)')

2x era cut: 2007-2016 +0.047 (t +2.26)  |  2017-2026 +0.004 (t +1.56)
  test of the DIFFERENCE (gap on an era dummy): -0.836 bps/day, HAC t -1.15 -- NOT distinguishable, so no decay is claimed
3x era cut: 2009-2016 -0.581 (t -2.80)  |  2017-2026 -0.563 (t -2.48)   difference -2.369 bps/day (t -0.30)

2x financing spread   0 bps: adv +0.021   200 bps: adv +0.025
2x trading cost       0 bps: adv +0.021   10 bps one-way: adv +0.029
(both arms hold nearly the same average exposure, so financing and cost
 hit them almost identically -- neither sweep can rescue or kill the result)


## 6. The 2008 stress UPRO never saw

> 💡 **In plain words** — UPRO launched after the crash. So we replay the construction on SPY through it.

In [7]:
print(f"SPY {R['st_start']} -> {R['st_end']} ({R['st_n']:,} days), ^IRX accrual as cash")
print(f"  2x monthly x{R['st_2x_mon']:.1f}   vs 2x daily x{R['st_2x_day']:.1f}  (never called)")
print(f"  3x monthly, no lender limit: x{R['st_3x_mon_free']:.1f}, "
      f"peak exposure {R['st_3x_mon_free_w']:.2f}x")
print(f"  3x monthly, 25% maintenance: x{R['st_3x_mon_25']:.2f}, called {R['st_3x_liq']}")
print(f"  3x daily                   : x{R['st_3x_day']:.1f}")
print(f"  4x monthly, no lender limit: x{R['st_4x_mon_free']:.2f} -- NEGATIVE EQUITY "
      f"{R['st_4x_liq']}")
print(f"  4x daily                   : x{R['st_4x_day']:.1f} -- cannot go below zero")

SPY 2004-01-05 -> 2026-06-30 (5,652 days), ^IRX accrual as cash
  2x monthly x37.6   vs 2x daily x27.9  (never called)
  3x monthly, no lender limit: x80.4, peak exposure 12.80x
  3x monthly, 25% maintenance: x0.96, called 2008-09-29
  3x daily                   : x33.8
  4x monthly, no lender limit: x0.00 -- NEGATIVE EQUITY 2008-10-24
  4x daily                   : x17.4 -- cannot go below zero


The 4x pair is the cleanest statement of what the daily reset is *for*. A constant-leverage fund's terminal value is bounded below by zero by construction; a drift-then-reset margin account's is not.

## 7. Live synthetic control — the machinery is unbiased

An AR(1) choppiness knob at **fixed total volatility**, so only the path shape changes. Liquidation disabled, seeds 943+.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from reset_freq import data, strategy as st
for tag, phi, ss in [('choppy  phi=-0.15', -0.15, 1.0),
                     ('trending phi=+0.15', 0.15, 1.0),
                     ('iid null phi= 0.00', -0.15, 0.0)]:
    d = [st.synthetic_detect(data.synthetic_daily(phi=phi, signal_strength=ss,
                                                  n_years=12, seed=943+s)[0])
         for s in range(4)]
    print(f"{tag}: gap {np.mean([x['mean_gap_bps'] for x in d]):+.3f} bps/day "
          f"(HAC t {np.mean([x['t_gap'] for x in d]):+.2f})  "
          f"ER slope {np.mean([x['chop_slope'] for x in d]):+.2f}")

choppy  phi=-0.15: gap +0.359 bps/day (HAC t +2.92)  ER slope -1.96


trending phi=+0.15: gap -0.402 bps/day (HAC t -1.93)  ER slope -2.65


iid null phi= 0.00: gap +0.035 bps/day (HAC t +0.21)  ER slope -2.27


Correct sign in both planted worlds, centred at zero on the null — and the ER slope stays strongly negative in all three, which is exactly the caveat of section 3, pinned as a live measurement (and as a unit test).

## Verdict

- **Signal — Mixed.** *Real on the return, absent on the Sharpe; positive at 2x, negative at 3x.* The reset-frequency effect on *returns* is real and clears the bar at 2x (+1.05 bps/day, HAC *t* = +2.79), with the theory-predicted conditional sign in both sleeves (slope *t* = -6.83 / -7.22) and a synthetic control that recovers it and stays quiet on the null. The claim under test nevertheless fails: risk-adjusted, the 2x advantage is +0.022 with CI [-0.010, +0.062], the lagged (tradable) form is *t* = +0.72, and the 3x advantage is -0.569 (*t* = -3.62) — or +0.003 even with an infinitely patient lender. The post-2017 half is weaker (+0.004 against +0.047) but the era difference is *t* = -1.15, so that is a hint, not a decay. Survivorship note: SSO and UPRO are the survivors, and the closed leveraged funds are not on this tape.
- **Tradability — Mirage.** Nothing bankable is on offer as a *reset* edge. The monthly reset sells uncontrolled leverage at an unchanged *average* leverage — peak 8.42x on a 2.98 mean, a call on 2011-08-08, negative equity at 4x in 2008 — and roughly half of its apparent advantage over the funds is their 1.21%/yr fee-and-tracking drag, which is a different study's question and is only yours if you borrow near the bill rate.